In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yfinance as yf
from sklearn.linear_model import RidgeCV
from sklearn.metrics import accuracy_score, mean_absolute_percentage_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_DIR = Path('model_artifacts')
MODEL_DIR.mkdir(exist_ok=True)

device


In [ ]:
def get_advanced_features(ticker="NVDA", start="2015-01-01"):
    df = yf.download(ticker, start=start, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))
    df['Vol_Rel'] = df['Volume'] / df['Volume'].rolling(20).mean()

    # Classifier target: will the price be higher in five trading days?
    df['Target'] = (df['Close'].shift(-5) > df['Close']).astype(int)

    return df.replace([np.inf, -np.inf], np.nan).dropna()

data = get_advanced_features("NVDA")
print(f"Data processed. Total rows: {len(data)}")
data.tail()


In [ ]:
SEQ_LEN = 30
feature_cols = ['Close', 'Returns', 'MA20', 'MA50', 'RSI', 'Vol_Rel']

scaler = RobustScaler()
scaled_data = scaler.fit_transform(data[feature_cols])

def create_sequences(values, target, seq_len):
    X, y = [], []
    for i in range(len(values) - seq_len):
        X.append(values[i:(i + seq_len)])
        y.append(target[i + seq_len])
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(scaled_data, data['Target'].values, SEQ_LEN)

X_tensor = torch.FloatTensor(X_seq).to(device)
y_tensor = torch.FloatTensor(y_seq).view(-1, 1).to(device)

print(f"Classifier input shape: {X_tensor.shape}")


In [ ]:
class StockSageModel(nn.Module):
    def __init__(self, n_features):
        super(StockSageModel, self).__init__()
        self.cnn = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3)
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.cnn(x))
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        x = self.relu(self.fc1(out[:, -1, :]))
        return self.sigmoid(self.fc2(x))

model = StockSageModel(len(feature_cols)).to(device)
model


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("Training direction classifier...")
for epoch in range(20):
    model.train()
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch + 1}/20], Loss: {loss.item():.4f}")

torch.save(model.state_dict(), 'model_artifacts/best_model.pth')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
metadata = {
    "feature_cols": feature_cols,
    "seq_len": SEQ_LEN,
    "framework": "pytorch",
    "direction_target_days": 5,
}
with open('model_artifacts/metadata.json', 'w') as f:
    json.dump(metadata, f)

print("Direction model saved to model_artifacts/best_model.pth")


In [ ]:
HORIZON_DAYS = {
    "1 Week": 5,
    "1 Month": 21,
    "3 Months": 63,
    "6 Months": 126,
}


def train_price_forecaster(df_feat, timeframe="1 Month"):
    """Train a per-ticker model that predicts forward log return, then convert it to price."""
    horizon_days = HORIZON_DAYS.get(timeframe, 21)
    feature_cols = ['Close', 'Returns', 'MA20', 'MA50', 'RSI', 'Vol_Rel']

    model_df = df_feat.copy()
    model_df['Forward_Log_Return'] = np.log(model_df['Close'].shift(-horizon_days) / model_df['Close'])
    model_df = model_df.replace([np.inf, -np.inf], np.nan).dropna()

    X = model_df[feature_cols]
    y = model_df['Forward_Log_Return']
    split_idx = max(int(len(model_df) * 0.8), len(model_df) - max(30, horizon_days))

    X_train, X_valid = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_valid = y.iloc[:split_idx], y.iloc[split_idx:]

    forecaster = Pipeline([
        ('scaler', RobustScaler()),
        ('model', RidgeCV(alphas=np.logspace(-4, 4, 25)))
    ])
    forecaster.fit(X_train, y_train)

    valid_pred = forecaster.predict(X_valid)
    actual_prices = model_df['Close'].iloc[split_idx:].to_numpy() * np.exp(y_valid.to_numpy())
    predicted_prices = model_df['Close'].iloc[split_idx:].to_numpy() * np.exp(valid_pred)
    mape = mean_absolute_percentage_error(actual_prices, predicted_prices)
    residual_std = np.std(y_valid.to_numpy() - valid_pred)

    latest_features = df_feat[feature_cols].tail(1)
    predicted_log_return = forecaster.predict(latest_features)[0]
    current_price = df_feat['Close'].iloc[-1]
    forecast_price = current_price * np.exp(predicted_log_return)
    interval_width = max(1.28 * residual_std, 0.01)

    return {
        "model": forecaster,
        "horizon_days": horizon_days,
        "current_price": current_price,
        "forecast_price": forecast_price,
        "expected_return_pct": (forecast_price / current_price - 1) * 100,
        "lower_price": current_price * np.exp(predicted_log_return - interval_width),
        "upper_price": current_price * np.exp(predicted_log_return + interval_width),
        "validation_mape_pct": mape * 100,
    }

price_forecast = train_price_forecaster(data, "1 Month")
price_forecast


In [ ]:
print(
    f"Current price: ${price_forecast['current_price']:.2f}\n"
    f"Forecast price: ${price_forecast['forecast_price']:.2f}\n"
    f"Expected move: {price_forecast['expected_return_pct']:+.2f}%\n"
    f"80% range: ${price_forecast['lower_price']:.2f} - ${price_forecast['upper_price']:.2f}\n"
    f"Validation MAPE: {price_forecast['validation_mape_pct']:.2f}%"
)

future_dates = pd.bdate_range(data.index[-1], periods=price_forecast['horizon_days'] + 1)[1:]
forecast_path = np.geomspace(
    price_forecast['current_price'],
    price_forecast['forecast_price'],
    num=price_forecast['horizon_days'] + 1
)[1:]

plt.figure(figsize=(12, 6))
plt.plot(data.index[-120:], data['Close'].tail(120), label='Historical Close')
plt.plot(future_dates, forecast_path, '--', label='Forecast')
plt.fill_between(
    future_dates,
    np.geomspace(price_forecast['current_price'], price_forecast['lower_price'], num=price_forecast['horizon_days'] + 1)[1:],
    np.geomspace(price_forecast['current_price'], price_forecast['upper_price'], num=price_forecast['horizon_days'] + 1)[1:],
    alpha=0.2,
    label='80% Forecast Range'
)
plt.title('NVDA Price Forecast')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()
